 # Ensambles paralelos homogeneos

Los métodos de conjunto paralelo entrenan cada estimador base independientemente de los demás, lo que significan que se pueden entrenar en paralelo. Dependiendo del conjunto de modelos base que utilicen se puede distinguir entre ensambles homogeneos y heterogeneos. Empezaremos revisando modelos homogenoes como lo son baggin y random forest.

Dados que los dos elementos esenciales de los modelos de ensamble son la diversidad y la agregación del modelo, enensambles homogeneos la diversidad se logra a través de random sampling (bagging), características o ambos, como lo hacen los random forest.

## Bagging: Bootstrap aggregating

Bagging, abreviación de bootstrap aggregating, fue introducido por Leo Breiman en 1996, y su nombre viene de la forma cómo se logra la diversidad del ensamble, a través del botstrap sampling o muestreo con remplazo, para luego hacer una agregación de los modelos.

El Bagging consiste en dos pasos: primero se realiza el muestreo con remplazo de subconjuntos del conjunto original dedatos, después se combinan para tener una unica predicción del ensamble.

Entonces, partimnos por generar los subconjuntos del conjunto original de entrnamiento, el cual se hace a terravés de remuestreo con remplazo, que es lo que se conoce como Bootstrap sampling. Naturalmente, esto genera dos subconjuntos:
1. Boostrap sample
2. Out-of-bag (OOB) sample: las que no estan en el subconjunto muestreado.

Podemos, entonces, entrenar diferentes estimadores base o modelos con cada bootstrap. Estos modelos se pueden evaluar internamente con el OOB, estimando un error de predicción que generalmente se conoce como error OOB o puntuación OOB. El error OOB promedio es una buena estimación del desempeño del conjunto general.

Utilicemos numpy para generar muestras con remplazo. Tomemos un conjunto de 50 observaciones, podemos generar bootstrap sample del mismo tamaño como sigue:

In [1]:
# bootstrap sample

import numpy as np

bag = np.random.choice(range(0,50),size=50,replace=True)
np.sort(bag)

array([ 0,  1,  1,  2,  2,  5,  6,  6,  7,  9, 10, 10, 10, 11, 11, 12, 13,
       14, 14, 17, 17, 18, 19, 20, 21, 25, 27, 28, 28, 30, 30, 30, 31, 32,
       33, 36, 37, 39, 40, 40, 43, 43, 43, 43, 43, 45, 45, 46, 48, 49])

In [2]:
# OOB sample

oob = np.setdiff1d(range(0,50),bag)
oob

array([ 3,  4,  8, 15, 16, 22, 23, 24, 26, 29, 34, 35, 38, 41, 42, 44, 47])

Nótese que el muestreo bootstrap contendrá aproximadamente el 63.2% del conjunto de datos, mientras que la muestra OOB contendrá el 36.8%. Esto se deduce del hecho de quemientras la probabilidad de elegir un registro es de $\frac{1}{n}$, la probabilidad de que sea parte delconjunto será de $1-\frac{1}{n}$, por lo que para $n$ puntos, tenemos que la probabilidad global de ser elegido:
$$
(1-\frac{1}{n})^{n} \approx e^{-1} \approx 0.368
$$

## Model agrregation

Habiendo entrenado diversos modelos independientes, debemos agregarlos en un estimador final, lo ual se puede hacerpot majority voting o moda estadística (en tareas de clasificación) o averaging (para tareas de regresión). De alguna manera, si los estimadores base son sólidos, la agregación funcionara como una suavización de los erroes de los estimadores base.



In [8]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# Inicializamos una semilla para reproducibilidad

rng = np.random.RandomState(seed=4190)

# La función aceptará parámetros como número de estimadores (árboles)
# profundidad y muestras de entrenamiento

def bagging_fit(X, y, n_estimators, max_depth = 5, max_samples = 200):
  n_samples = len(y) # observaciones disponibles n

  # Creamos una lista de estimadores base no entrenados
  estimators = [DecisionTreeClassifier(max_depth = max_depth) for _ in range(n_estimators)]

  # ahora, por cada elemento de la lista, muestreamo y entrenamos
  for tree in estimators:
    # Generamos un bootstrap sample (selección de los índices de las muestras)
    bag = rng.choice(n_samples, max_samples, replace = True)
    # Entrenamos cada arbol
    tree.fit(X[bag,:],y[bag])

  return estimators

In [7]:
# Usaremos la moda como método de agregación

from scipy.stats import mode

# obtenemos las predicciones individuales

def bagging_predict(X, estimators):
  all_predictions = np.array([tree.predict(X) for tree in estimators])

  y_pred, _ = mode(all_predictions, axis = 0, keepdims = False)

  return np.squeezee(ypred)